<!-- formatted -->
# RAG from Scratch

This is a series I am starting for myself, in order to learn and understand **RAG** better.

It is highly motivated by the YouTube series [_RAG From Scratch_](https://youtu.be/sVcwVQRHIc8?si=oOds6Njaj0WTOvq3), which I recently completed. I was really impressed by how well the topics were chosen and explained. The series also linked a GitHub repo with all the examples used — which I think is a great go-to reference for the future.

So I decided to **rewrite the complete code examples myself** — with:

- my own understanding
- my own words
- my own examples

This will not only help me understand RAG better, but will also serve as my personal notebook for the basics down the line.

So here I am. **Let's start RAG.** 🚀

<!-- formatted -->
Let's start with the full form of RAG — **Retrieval-Augmented Generation**. Let's dissect this term, starting with the _"Generation"_ part.

**How I like to think of an LLM:**

- It's like a super intelligent person who has read all the books, all the internet — knows everything you name it.
- But there's a catch: all of that knowledge is **PUBLIC**. Remember the word _public_ here.
- Not every knowledge is public. So even someone this intelligent can't help you with the information which only you have — which is a Meh!!

**This is where RAG comes in:**

- If there's a way to safely pass this private info to that intelligent person _inside our question_, then he can understand it really well, combine it with his own skills, and give you a perfect answer.
- So RAG basically helps you **AUGMENT** your query with your private info, allowing the LLM to generate more informed and accurate responses.
- And where does this private info live? In your brain — or for a company, in their company DB.

**Putting it together:**

> The whole process of getting the private, relevant data, augmenting it with your query, and getting a very grounded reply — that is RAG!

<!-- formatted -->
A complete RAG system can be said to be made of **4 main processes**:

1. **Loaders**: Loaders are responsible for loading the data from various sources — the internet, files, folders, etc. The goal is to get the data into a format that can be processed further.
2. **Chunking**: Once the data is loaded, we need to break it down into smaller, manageable pieces.
3. **Embeddings**: Once the data is chunked, we need to convert it into a numerical format that can be understood by machine learning models. This is done using embeddings, which are vector representations of the data. Embeddings capture the semantic meaning of the data, allowing us to compare and retrieve relevant information.
4. **Retrieval**: Finally, we need to retrieve the relevant chunks of data based on the user's query. This is done using similarity search, where we compare the embeddings of the query with the embeddings of the chunks and return the most relevant ones.

```
Loaders → Chunking → Embeddings → Retrieval
```

Let's start implementing the basic RAG pipeline.

<!-- formatted -->
## Loaders

In [2]:


from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=[
        "https://en.wikipedia.org/wiki/Artificial_intelligence",
        "https://en.wikipedia.org/wiki/Generative_AI",
        "https://de.wikipedia.org/wiki/Large_Language_Model"
    ]
)

docs = loader.load()

In [8]:
print(f"Loaded {len(docs)} documents")

Loaded 3 documents


<!-- formatted -->
## Chunking

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

split_docs = text_splitter.split_documents(docs)

In [10]:
print(f"Split into {len(split_docs)} chunks")

Split into 1115 chunks


<!-- formatted -->
## Embeddings & Inserting into Vector Store

The embedding part gets a bit hidden in LangChain — we just pass the embedding model to the vector store's init method, and it takes care of embedding and inserting the chunks into the vector store.

In [23]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
import os

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

embedding_model = OpenAIEmbeddings(
    openai_api_base="https://openrouter.ai/api/v1",
    model="openai/text-embedding-ada-002",
    openai_api_key=OPENROUTER_API_KEY
)

# Here the split_docs are first embedded using the model we specified, and then inserted into the vector store.
vector_store = Chroma.from_documents(
    documents=split_docs,
    embedding=embedding_model,
    collection_name="basic-rag-overview",
)

retriever = vector_store.as_retriever()

retriever.invoke("What is LLM?")


[Document(id='ef2ddb26-e18a-4e10-911d-141f1f3ce933', metadata={'source': 'https://de.wikipedia.org/wiki/Large_Language_Model', 'language': 'de', 'title': 'Large Language Model – Wikipedia'}, page_content='Bei LLMs handelt es sich dabei um eine Reihe von Techniken (Algorithmen) und anderen Softwareartefakten (Frameworks und Programmbibliotheken) im Bereich moderner künstlicher Intelligenz (KI), die seit etwa Mitte der 2010er Jahre existieren und seit den 2020er Jahren vermehrt auf der Basis des Cloud Computing bereitgestellt werden. Dabei kommen Serversysteme mit KI-optimierten Mikrochips zum Einsatz. Einige Softwareanwendungen erlauben die lokale Ausführung von LLMs, welche jedoch durch die'),
 Document(id='24d8688b-6b88-4ef5-be98-c856a2b12b0a', metadata={'title': 'Large Language Model – Wikipedia', 'source': 'https://de.wikipedia.org/wiki/Large_Language_Model', 'language': 'de'}, page_content='Bei LLMs handelt es sich dabei um eine Reihe von Techniken (Algorithmen) und anderen Softwar

In [33]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter

prompt_template = """You are a helpful assistant.
Use the following pieces of context to answer the question. Answer ONLY if the answer can be found in the context below. If the answer is not contained within the text below, say "I don't know"

If the context contains sources, always make sure to add sources at appropriate place in your reply.

Context:
{context}

Question:
{question}
"""

llm = ChatGroq(model="qwen/qwen3-32b", temperature=0.0, reasoning_format="hidden")

prompt = PromptTemplate.from_template(prompt_template)

# This very interesting. Check it closely. The itemgetter is used to get the question from the input dict, and then it is passed to the retriever. The retriever returns the context, which is then passed to the prompt. The prompt is then passed to the llm, which returns the answer. The answer is then parsed using the StrOutputParser.
chain = (
        {
            "context": itemgetter("question") | retriever,
            "question": itemgetter("question")
        }
        | prompt
        | llm
        | StrOutputParser()
)

chain.invoke({"question": "Tell me various model training techniques in pointwise, easy to understand way?"})

# TODO: See how to link sources. Play around sources

'Here are some model training techniques explained in a simple way, based on the provided context:\n\n1. **Generative Adversarial Networks (GANs)**  \n   - A technique where two models (generator and discriminator) compete during training. The generator creates data (e.g., images), while the discriminator evaluates its authenticity. This adversarial process improves the generator\'s ability to produce realistic outputs.  \n   - *Source: [Generative AI - Wikipedia](https://en.wikipedia.org/wiki/Generative_AI)*  \n\n2. **Word Embeddings**  \n   - A method to represent words as numerical vectors, capturing their meanings and relationships. This helps models understand language better during training.  \n   - *Source: [Artificial intelligence - Wikipedia](https://en.wikipedia.org/wiki/Artificial_intelligence)*  \n\n3. **Transformers**  \n   - A deep learning architecture using "attention mechanisms" to process sequential data (e.g., text). Transformers enable models to focus on relevant pa